In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/

In [2]:
x = torch.tensor([1,2,3,4,5])
x 

tensor([1, 2, 3, 4, 5])

In [3]:
words = open('names.txt','r').read().splitlines()

In [4]:
print(len(words),words[:3])

32033 ['emma', 'olivia', 'ava']


In [5]:
# creatign the lookup tables
k = words[:4]
chars = sorted(list(set(''.join(words))))

stoi = { s : i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [6]:
# dataset creation

block_size = 3   #context length
X, Y = [], []    # input and labels

for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context),'--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [7]:
X.shape, X.dtype, Y.shape, Y.dtype, X

(torch.Size([32, 3]),
 torch.int64,
 torch.Size([32]),
 torch.int64,
 tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1],
         [ 0,  0,  0],
         [ 0,  0,  9],
         [ 0,  9, 19],
         [ 9, 19,  1],
         [19,  1,  2],
         [ 1,  2,  5],
         [ 2,  5, 12],
         [ 5, 12, 12],
         [12, 12,  1],
         [ 0,  0,  0],
         [ 0,  0, 19],
         [ 0, 19, 15],
         [19, 15, 16],
         [15, 16,  8],
         [16,  8,  9],
         [ 8,  9,  1]]))

In [8]:
C = torch.randn((27,2))
C

tensor([[-2.8084, -0.9343],
        [-0.6041, -0.8328],
        [-1.1675, -0.4570],
        [ 0.9705,  0.2183],
        [-0.5302, -0.7851],
        [-0.4816, -1.7497],
        [-0.1103, -0.3316],
        [ 1.1356, -1.7710],
        [-0.4381, -2.1753],
        [ 0.1312, -1.0259],
        [-0.5173,  1.0166],
        [-0.0044, -1.0221],
        [-0.2550,  0.1424],
        [-0.2784,  1.7401],
        [-1.2504,  0.8439],
        [-0.0371, -0.4243],
        [ 0.1938,  1.0475],
        [-1.5370, -1.3074],
        [ 0.4676, -0.1368],
        [-0.6361,  0.9192],
        [ 0.5239,  0.6323],
        [-0.9797,  1.0006],
        [ 0.7673, -0.9917],
        [ 0.6024, -0.6136],
        [-0.5777, -0.7659],
        [ 1.1704,  0.1247],
        [ 2.4213,  1.8140]])

In [9]:
C[X].shape , C[X]

(torch.Size([32, 3, 2]),
 tensor([[[-2.8084, -0.9343],
          [-2.8084, -0.9343],
          [-2.8084, -0.9343]],
 
         [[-2.8084, -0.9343],
          [-2.8084, -0.9343],
          [-0.4816, -1.7497]],
 
         [[-2.8084, -0.9343],
          [-0.4816, -1.7497],
          [-0.2784,  1.7401]],
 
         [[-0.4816, -1.7497],
          [-0.2784,  1.7401],
          [-0.2784,  1.7401]],
 
         [[-0.2784,  1.7401],
          [-0.2784,  1.7401],
          [-0.6041, -0.8328]],
 
         [[-2.8084, -0.9343],
          [-2.8084, -0.9343],
          [-2.8084, -0.9343]],
 
         [[-2.8084, -0.9343],
          [-2.8084, -0.9343],
          [-0.0371, -0.4243]],
 
         [[-2.8084, -0.9343],
          [-0.0371, -0.4243],
          [-0.2550,  0.1424]],
 
         [[-0.0371, -0.4243],
          [-0.2550,  0.1424],
          [ 0.1312, -1.0259]],
 
         [[-0.2550,  0.1424],
          [ 0.1312, -1.0259],
          [ 0.7673, -0.9917]],
 
         [[ 0.1312, -1.0259],
          [ 0.7

In [10]:
emb = C[X]
emb.shape 

torch.Size([32, 3, 2])

In [11]:
w1 = torch.randn(6, 100)    # as we have no_of_inputs = 6
b1 = torch.rand(100)

In [12]:
# emb @ w1 + b1   ---> our AIM but 

torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]],1).shape

torch.Size([32, 6])

In [25]:
torch.cat(torch.unbind(emb,1),1).shape    # concatenation uses extra memory

torch.Size([32, 6])

In [16]:
a = torch.arange(20)
a.shape

torch.Size([20])

In [20]:
a.view(2,10)   # no extra memory is used 

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]])

In [23]:
a.storage()  # always represented in memory like one-dimensional vector

 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
 18
 19
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 20]

In [27]:
emb.view(32,6) == torch.cat(torch.unbind(emb,1),1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [31]:
h = torch.tanh(emb.view(-1,6) @ w1 + b1)
h.shape

torch.Size([32, 100])

In [32]:
# final layer of NN
w2 = torch.randn(100,27)
b2 = torch.randn(27)

In [34]:
logits = h @ w2 + b2
logits.shape

torch.Size([32, 27])

In [37]:
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)   # normalizing the sum

In [38]:
probs[0].sum()

tensor(1.0000)

In [39]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [43]:
loss = -probs[torch.arange(32),Y].log().mean()
loss

tensor(15.1726)

In [47]:
# Readable version

g = torch.Generator().manual_seed(2147483647)
c = torch.randn((27,2), generator=g)

w1 = torch.randn((6,100),generator=g)
b1 = torch.randn(100,generator=g)

w2 = torch.randn((100,27),generator=g)
b2 = torch.randn(27,generator=g)

In [52]:
# calculation

emb = C[X]
h = torch.tanh(emb.view(-1,6) @ w1 + b1)
logits = h @ w2 + b2

# counts = logits.exp()
# probs = counts / counts.sum(1, keepdims=True)   # normalizing the sum
# loss = -probs[torch.arange(32),Y].log().mean()

F.cross_entropy(logits, Y)

tensor(17.8659)

In [53]:
loss

tensor(17.8659)